# Deliverable 3 — Single-Stock Strategy Evaluation (NVDA)

This notebook backtests five trading strategies on NVIDIA (NVDA) over the full 5-year dataset
(April 2021 – April 2026). All strategies operate at daily closing prices with no slippage,
no short selling, and no leverage. The portfolio is held as cash during each strategy's
warmup period.

**Strategies tested:**
1. **Momentum (20d)** — Buy if 20-day return is positive; hold cash otherwise.
2. **Mean Reversion (20d)** — Buy on a z-score dip below -1; exit above +1.
3. **Dual Momentum (20/60d)** — Buy only when BOTH 20-day AND 60-day returns are positive.
4. **Breakout (252d high / 20d low)** — Enter on a 52-week high; exit on a 20-day low.
5. **Trend + MR Combo** — Requires 60-day uptrend; buys dips with z-score < -0.5.

In [ ]:
%matplotlib inline
import os, sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Run from project root regardless of where Jupyter was launched
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

from backtester import load_prices, run_backtest
from backtester.metrics import format_metrics
from strategies.momentum import Momentum
from strategies.mean_reversion import MeanReversion
from strategies.single_stock import DualMomentum, Breakout, TrendMeanReversionCombo

In [ ]:
prices = load_prices('data/nasdaq100_daily_5y.csv')
print(f'Loaded: {prices.shape[0]} trading days × {prices.shape[1]} tickers')
print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')

In [ ]:
# Run all five strategies frictionless — fresh instance per run for stateful strategies
specs = [
    ('Momentum (20d)',              lambda: Momentum(ticker='NVDA', lookback=20)),
    ('Mean Reversion (20d)',        lambda: MeanReversion(ticker='NVDA', window=20)),
    ('Dual Momentum (20/60d)',      lambda: DualMomentum(ticker='NVDA', short_window=20, long_window=60)),
    ('Breakout (252d/20d)',         lambda: Breakout(ticker='NVDA', breakout_window=252, breakdown_window=20)),
    ('Trend + MR Combo',           lambda: TrendMeanReversionCombo(ticker='NVDA')),
]

results = []
for label, factory in specs:
    r = run_backtest(prices, factory(), transaction_cost=0.0, name_override=label)
    results.append(r)

## Combined NAV Comparison

All strategies start with NAV = 1.0. The flat portion at the beginning of each curve is the
warmup period during which the portfolio holds cash.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
for r in results:
    nav = r['nav_series']
    ax.plot(nav.index, nav.values, label=r['strategy_name'], linewidth=1.5)
ax.set_title('NVDA Single-Stock Strategy Comparison (frictionless)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('NAV (normalized to 1.0)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=9, frameon=True)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Metrics Table

In [ ]:
rows = []
for r in results:
    m = r['metrics']
    rows.append({
        'Strategy':         r['strategy_name'],
        'Cum. Return':      f"{m['cumulative_return']:+.2%}",
        'Ann. Volatility':  f"{m['annualized_volatility']:.2%}",
        'Sharpe Ratio':     f"{m['sharpe_ratio']:.2f}",
        'Max Drawdown':     f"{-m['max_drawdown']:+.2%}",
        'Win Rate':         f"{m['win_rate']:.2%}",
    })
pd.DataFrame(rows).set_index('Strategy')

## Results Interpretation

**Momentum** leads on both cumulative return (+369%) and Sharpe (1.00). NVDA was in a sustained
multi-year uptrend driven by AI infrastructure demand — momentum strategies that simply follow
the direction of price are well-suited to this regime.

**Dual Momentum** achieves a nearly identical Sharpe (0.99) with a meaningfully lower max
drawdown (-36% vs -50%). Requiring confirmation across two timeframes filters out short-term
whipsaws during corrections, at the cost of slightly delayed entries.

**Mean Reversion** underperforms as expected (Sharpe 0.79). On a stock with NVDA's structural
uptrend, betting against the direction each time it moves significantly up is systematically
wrong. The strategy still profits overall because NVDA's upward bias is so strong that even
oversold bounces are profitable.

**Breakout** shows the lowest volatility (25.6%) and smallest max drawdown (-33%) of the trend-
following group. Its 252-day warmup means it misses NVDA's 2021 run entirely, entering only
in April 2022. The entry-on-new-high filter keeps it out of declining markets but also means
it often buys after most of a move has already occurred.

**Trend + MR Combo** produces the lowest return (+15%) and Sharpe (0.25). The strategy
requires both a 60-day uptrend AND a z-score dip below -0.5 simultaneously. In a strong bull
market, dips within the trend are shallow and brief — the z-score threshold is rarely triggered,
so the strategy spends most of the period in cash.

---
## Individual Strategy NAV Curves

### Strategy 1: Momentum (20d)

Allocates fully to NVDA when the 20-day trailing return is positive; holds cash otherwise.
This is the simplest trend-following rule and serves as the performance baseline.

In [ ]:
r = results[0]
fig, ax = plt.subplots(figsize=(12, 4))
nav = r['nav_series']
ax.plot(nav.index, nav.values, color='steelblue', linewidth=1.5)
ax.set_title(r['strategy_name'], fontsize=12, fontweight='bold')
ax.set_ylabel('NAV')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
print(format_metrics(r['metrics']))

### Strategy 2: Mean Reversion (20d)

Computes the rolling 20-day z-score of NVDA's price. Buys when z < -1 (oversold); sells
to cash when z > 1 (overbought). Maintains the previous position in the neutral zone to
avoid whipsawing. This strategy fights the trend and is expected to underperform on NVDA.

In [ ]:
r = results[1]
fig, ax = plt.subplots(figsize=(12, 4))
nav = r['nav_series']
ax.plot(nav.index, nav.values, color='darkorange', linewidth=1.5)
ax.set_title(r['strategy_name'], fontsize=12, fontweight='bold')
ax.set_ylabel('NAV')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
print(format_metrics(r['metrics']))

### Strategy 3: Dual Momentum (20d / 60d)

Requires BOTH the 20-day and 60-day returns to be positive before investing. One positive
window is not enough — both must confirm the trend. This extension of pure momentum reduces
false signals during choppy markets at the cost of slightly delayed entries.

In [ ]:
r = results[2]
fig, ax = plt.subplots(figsize=(12, 4))
nav = r['nav_series']
ax.plot(nav.index, nav.values, color='seagreen', linewidth=1.5)
ax.set_title(r['strategy_name'], fontsize=12, fontweight='bold')
ax.set_ylabel('NAV')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
print(format_metrics(r['metrics']))

### Strategy 4: Breakout (52-week high / 20-day low)

Enters when NVDA closes at a new 252-day (52-week) high — a structural breakout above
resistance. Exits when the price falls to a 20-day low — a breakdown of near-term support.
The 252-day warmup means this strategy only begins trading in April 2022.

In [ ]:
r = results[3]
fig, ax = plt.subplots(figsize=(12, 4))
nav = r['nav_series']
ax.plot(nav.index, nav.values, color='mediumpurple', linewidth=1.5)
ax.set_title(r['strategy_name'], fontsize=12, fontweight='bold')
ax.set_ylabel('NAV')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
print(format_metrics(r['metrics']))

### Strategy 5: Trend + Mean Reversion Combo

A hybrid: first checks whether NVDA is in a 60-day uptrend. Only if confirmed does it
then look for a mean-reversion dip entry (z-score below -0.5). The intent is to buy
pullbacks within bull markets. In practice, the strict dual condition — uptrend AND
z < -0.5 simultaneously — keeps this strategy largely in cash.

In [ ]:
r = results[4]
fig, ax = plt.subplots(figsize=(12, 4))
nav = r['nav_series']
ax.plot(nav.index, nav.values, color='crimson', linewidth=1.5)
ax.set_title(r['strategy_name'], fontsize=12, fontweight='bold')
ax.set_ylabel('NAV')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
print(format_metrics(r['metrics']))